# 00 — Data Exploration: ArXiv + SciTLDR

Exploratory Data Analysis on both datasets used for fine-tuning.
This notebook is **CPU-safe** — no model loading required.

**Datasets:**
- `ccdv/arxiv-summarization` — long scientific papers with structured abstracts
- `allenai/scitldr` (AIC split) — short TLDR summaries of CS papers

**Sections:**
1. Load & inspect both datasets
2. Length distributions
3. Compression ratio analysis
4. Sample examples
5. Dataset comparison table

In [ ]:
# Install minimal deps if not already installed
import subprocess, sys
pkgs = ['datasets', 'matplotlib', 'seaborn', 'pandas', 'numpy']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Imports OK')

## 1. Load Datasets

In [ ]:
print('Loading ArXiv summarization dataset ...')
arxiv_ds = load_dataset('ccdv/arxiv-summarization', trust_remote_code=True)
print(f'ArXiv splits: {list(arxiv_ds.keys())}')
print(f'  Train samples: {len(arxiv_ds["train"]):,}')
print(f'  Val samples:   {len(arxiv_ds["validation"]):,}')
print(f'  Test samples:  {len(arxiv_ds["test"]):,}')
print(f'  Columns: {arxiv_ds["train"].column_names}')

In [ ]:
print('Loading SciTLDR (AIC split) ...')
scitldr_ds = load_dataset('allenai/scitldr', 'AIC', trust_remote_code=True)
print(f'SciTLDR splits: {list(scitldr_ds.keys())}')
print(f'  Train samples: {len(scitldr_ds["train"]):,}')
print(f'  Val samples:   {len(scitldr_ds["validation"]):,}')
print(f'  Test samples:  {len(scitldr_ds["test"]):,}')
print(f'  Columns: {scitldr_ds["train"].column_names}')

## 2. Sample Examples

In [ ]:
print('=== ArXiv Sample ===')
sample = arxiv_ds['train'][0]
print(f'Source (first 500 chars):\n{sample["article"][:500]}\n')
print(f'Target (abstract):\n{sample["abstract"][:300]}')

In [ ]:
print('=== SciTLDR Sample ===')
sample = scitldr_ds['train'][0]
src = ' '.join(sample['source']) if isinstance(sample['source'], list) else sample['source']
tgt = sample['target'][0] if isinstance(sample['target'], list) else sample['target']
print(f'Source (first 500 chars):\n{src[:500]}\n')
print(f'Target (TLDR):\n{tgt}')

## 3. Length Distributions

In [ ]:
# Sample 2000 examples for fast analysis
N = 2000
rng = np.random.default_rng(42)

arxiv_sample = arxiv_ds['train'].shuffle(seed=42).select(range(N))
arxiv_src_lens = [len(x['article'].split()) for x in arxiv_sample]
arxiv_tgt_lens = [len(x['abstract'].split()) for x in arxiv_sample]

scitldr_all = scitldr_ds['train']
sci_idx = rng.integers(0, len(scitldr_all), size=min(N, len(scitldr_all)))
sci_sample = scitldr_all.select(sci_idx.tolist())

def flatten_src(x):
    src = ' '.join(x['source']) if isinstance(x['source'], list) else x['source']
    tgt = x['target'][0] if isinstance(x['target'], list) else x['target']
    return len(src.split()), len(tgt.split())

sci_lens = [flatten_src(x) for x in sci_sample]
scitldr_src_lens = [s for s, _ in sci_lens]
scitldr_tgt_lens = [t for _, t in sci_lens]

print(f'ArXiv   — avg source: {np.mean(arxiv_src_lens):.0f} words | avg target: {np.mean(arxiv_tgt_lens):.0f} words')
print(f'SciTLDR — avg source: {np.mean(scitldr_src_lens):.0f} words | avg target: {np.mean(scitldr_tgt_lens):.0f} words')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Length Distributions — ArXiv vs SciTLDR', fontsize=14, fontweight='bold')

# ArXiv source
axes[0, 0].hist(arxiv_src_lens, bins=50, color='#6366f1', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('ArXiv — Source Length (words)')
axes[0, 0].set_xlabel('Words'); axes[0, 0].set_ylabel('Count')

# ArXiv target
axes[0, 1].hist(arxiv_tgt_lens, bins=40, color='#a78bfa', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('ArXiv — Target Length (words)')
axes[0, 1].set_xlabel('Words')

# SciTLDR source
axes[1, 0].hist(scitldr_src_lens, bins=50, color='#10b981', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('SciTLDR — Source Length (words)')
axes[1, 0].set_xlabel('Words'); axes[1, 0].set_ylabel('Count')

# SciTLDR target
axes[1, 1].hist(scitldr_tgt_lens, bins=30, color='#34d399', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('SciTLDR — Target Length (words)')
axes[1, 1].set_xlabel('Words')

plt.tight_layout()
plt.savefig('docs/length_distributions.png', bbox_inches='tight')
plt.show()
print('Saved to docs/length_distributions.png')

## 4. Compression Ratio Analysis

In [ ]:
arxiv_ratios = [s / max(t, 1) for s, t in zip(arxiv_src_lens, arxiv_tgt_lens)]
scitldr_ratios = [s / max(t, 1) for s, t in zip(scitldr_src_lens, scitldr_tgt_lens)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(arxiv_ratios, bins=50, alpha=0.65, label=f'ArXiv (median={np.median(arxiv_ratios):.1f}x)', color='#6366f1')
ax.hist(scitldr_ratios, bins=50, alpha=0.65, label=f'SciTLDR (median={np.median(scitldr_ratios):.1f}x)', color='#10b981')
ax.set_xlabel('Compression Ratio (source / target words)')
ax.set_ylabel('Count')
ax.set_title('Compression Ratio Distribution')
ax.legend()
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()

## 5. Dataset Comparison Summary

In [ ]:
summary_df = pd.DataFrame({
    'Dataset': ['ArXiv', 'SciTLDR'],
    'Train Size': [f'{len(arxiv_ds["train"]):,}', f'{len(scitldr_ds["train"]):,}'],
    'Avg Source (words)': [f'{np.mean(arxiv_src_lens):.0f}', f'{np.mean(scitldr_src_lens):.0f}'],
    'Avg Target (words)': [f'{np.mean(arxiv_tgt_lens):.0f}', f'{np.mean(scitldr_tgt_lens):.0f}'],
    'Median Compression': [f'{np.median(arxiv_ratios):.1f}x', f'{np.median(scitldr_ratios):.1f}x'],
    'Domain': ['Scientific (all fields)', 'CS/ML papers'],
    'Summary Style': ['Structured abstract', 'Short TLDR (1-2 sentences)'],
})
print(summary_df.to_string(index=False))
summary_df.to_csv('docs/dataset_comparison.csv', index=False)
print('\nSaved to docs/dataset_comparison.csv')

## 6. Next Steps

- ✅ Both datasets loaded and inspected
- ✅ Length distributions understood
- → **Run `notebooks/01_training.ipynb`** to fine-tune Mistral-7B